# 🌑 LunarSight — Notebook 02: Despeckling Training

**Agent 2**: Train the Complex-Valued CNN (CV-CNN) autoencoder to remove
speckle noise while preserving polarimetric phase.

⚡ **GPU Required**: This notebook should be run on Colab with a T4 GPU.

---

In [ ]:
# === Setup ===
import os, torch
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Lunar-Sight'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {REPO_DIR}
os.chdir(os.path.join(REPO_DIR, 'Lunar-Sight'))
!pip install -q -r requirements_colab.txt

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === Configuration ===
import yaml, logging
logging.basicConfig(level=logging.INFO)

with open('config/mission_config.yaml') as f:
    mission_config = yaml.safe_load(f)
with open('config/training_config.yaml') as f:
    train_config = yaml.safe_load(f)

# Paths
TENSOR_PATH = 'outputs/agent1/co_registered_tensor.npy'  # From notebook 01
CHECKPOINT_DIR = '/content/drive/MyDrive/LunarSight/checkpoints/agent2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Checkpoint dir: {CHECKPOINT_DIR}')

In [ ]:
# === Create Dataset & DataLoaders ===
from agent2_despeckling.dataset import create_data_loaders

desp_cfg = train_config.get('despeckling', {})
train_loader, val_loader, dataset = create_data_loaders(
    tensor_path=TENSOR_PATH,
    patch_size=desp_cfg.get('patch_size', 128),
    stride=desp_cfg.get('stride', 64),
    batch_size=desp_cfg.get('batch_size', 8),
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# === Create Model ===
from agent2_despeckling.cv_cnn_model import create_model

model = create_model(
    in_channels=dataset.n_channels,
    base_filters=desp_cfg.get('base_filters', 32),
    dropout=desp_cfg.get('dropout', 0.1),
)

In [ ]:
# === Train ===
from agent2_despeckling.train import train_despeckling

history = train_despeckling(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,
)

print(f"Best val loss: {history['best_val_loss']:.6f}")
print(f"Final epoch: {history['final_epoch']}")

In [ ]:
# === Plot Training Curves ===
import matplotlib.pyplot as plt

h = history['history']
plt.figure(figsize=(10, 5))
plt.plot(h['epoch'], h['train_loss'], label='Train')
plt.plot(h['epoch'], h['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CV-CNN Despeckling Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# === Run Inference on Full Scene ===
import numpy as np
from agent2_despeckling.inference import run_inference, verify_phase_integrity

raw_tensor = np.load(TENSOR_PATH)
despeckled = run_inference(model, raw_tensor)

# Verify phase preservation
stats = verify_phase_integrity(raw_tensor, despeckled)
print(f"Phase integrity: {stats}")

# Save
os.makedirs('outputs/agent2', exist_ok=True)
np.save('outputs/agent2/despeckled_tensor.npy', despeckled)
print('Saved despeckled tensor')